In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/lopezjatjat10@gmail.com/sportsbar-etl-proj/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')


In [0]:

df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source};')
display(df_silver)

In [0]:
df_silver.printSchema()

In [0]:
df_gold = df_silver.select('product_code', 'product_id', 'division', 'category', 'product', 'variant')

display(df_gold.limit(10))

In [0]:
df_gold.write\
  .format('delta')\
  .mode('overwrite')\
  .option('delta.enableChangeDataFeed', 'true')\
  .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

In [0]:
#parent table
df_parent_product = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.dim_{data_source}')

#child table from sportsbar
df_child_product = spark.sql(f' SELECT product_code, division, category, product, variant FROM {catalog}.{gold_schema}.sb_dim_{data_source}')

display(df_child_product)

In [0]:
df_parent_product.alias('target').merge(
    source=df_child_product.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
     values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

In [0]:
table = spark.sql(f'SELECT * FROM {catalog}.{gold_schema}.dim_{data_source}')

display(table)